# 22 · Validación adicional de Qwen3-Embedding-4B

**Prueba de robustez, no certificación de superioridad ni evaluación humana.**
Reutiliza los 519 embeddings documentales y el índice existentes. No cambia consultas,
anotaciones, índices ni resultados anteriores. Ejecutar con `entorno_tesis`, desde
`07_RAG/Pruebas_RAG/`, con los otros modelos descargados de la GPU.

Se verifica la recuperación guardada, se vuelven a codificar **24 consultas originales una vez**,
y se ejecutan **60 consultas adicionales**: 24 reformulaciones, 24 con un error tipográfico,
8 formulaciones de cuatro necesidades nuevas y 4 controles de alcance. Los nuevos temas y
sus cantidades son decisiones operativas, no un tamaño muestral validado. Las variantes no
son necesidades independientes. Los controles **no tienen irrelevancia confirmada**.

Salida única y reanudable: `validacion_qwen3_4b_v1.json`.
La libreta no ejecuta Gemma, no ajusta prompts ni umbrales y no elige un ganador nuevo.

## 1. Entradas y procedencia
Se aceptan los dos nombres usados antes para las anotaciones, pero solo se carga la copia
cuya huella coincide con la evaluación. Si falta una entrada, se indica el archivo exacto.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, re, gc, platform, sys, time
import numpy as np
import pandas as pd
from IPython.display import display

VERSION = "VALIDACION_QWEN3_4B_V1"
SALIDA = "validacion_qwen3_4b_v1.json"
PERMITIR_CAMBIO_ENTORNO = False
SOLO_CACHE_LOCAL = True
TOL_SCORE = 2e-6


def exigir(condicion, mensaje):
    if not condicion:
        raise ValueError(mensaje)


def firma(obj):
    return hashlib.sha256(json.dumps(obj, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False).encode("utf-8")).hexdigest()


def sha(ruta):
    return hashlib.sha256(ruta.read_bytes()).hexdigest()


def localizar(nombres, carpetas, esperado=None):
    candidatos = list(dict.fromkeys(p / n for p in carpetas for n in nombres if (p / n).is_file()))
    validos = [p for p in candidatos if esperado is None or sha(p) == esperado]
    if not validos:
        vistos = "\n".join(f"{p}: {sha(p)}" for p in candidatos) or "Ninguno encontrado."
        raise FileNotFoundError(f"Se necesita: {', '.join(nombres)}.\n{vistos}\n"
                                f"SHA-256 esperado: {esperado or 'no fijado'}")
    exigir(len({sha(p) for p in validos}) == 1, f"Copias distintas de {nombres}; conserva una versión inequívoca.")
    return validos[0]


carpetas = list(dict.fromkeys(p / s for p in (Path.cwd(), *Path.cwd().parents)
    for s in (".", "07_RAG/Pruebas_RAG", "Pruebas_RAG")))
carpeta_seleccion = next((p for p in carpetas if (p / "seleccion_recuperador_v1.json").is_file()), None)
if carpeta_seleccion is None:
    raise FileNotFoundError("Falta seleccion_recuperador_v1.json en 07_RAG/Pruebas_RAG/.")
r_s = localizar(["seleccion_recuperador_v1.json"], [carpeta_seleccion])
CARPETA = r_s.parent
s_pack = json.loads(r_s.read_text(encoding="utf-8"))
S = s_pack["seleccion"]
exigir(firma(S) == s_pack["firma_seleccion"], "Firma de selección incorrecta.")
exigir(S["recomendacion"]["sistema"] == "qwen3_4b", "Esta libreta está limitada a Qwen3-4B.")
r_e = localizar(["evaluacion_recuperacion_v1.json"], [CARPETA], S["fuentes"]["evaluacion"]["sha256"])
r_d = localizar(["densa_resultados_v1.json"], [CARPETA], S["fuentes"]["densa"]["sha256"])
E = json.loads(r_e.read_text(encoding="utf-8"))["resultados"]
D = json.loads(r_d.read_text(encoding="utf-8"))
r_a = localizar(["pertinencia_completa_N01_N12.json", "conjunto_pertinencia_anotado_v1.json"],
               [CARPETA], E["fuentes"]["anotaciones"]["sha256"])
A = json.loads(r_a.read_text(encoding="utf-8"))
Q = D["sistemas"]["qwen3_4b"]
CFG = Q["condicion"]
exigir(Q["firma_condicion"] == S["firma_condicion_documentada"] and CFG == S["configuracion_evaluada"], "Condiciones incompatibles.")
exigir(firma(A["base"]) == A["firma_conjunto"] == S["firma_conjunto"], "Anotaciones incompatibles.")
exigir(firma(Q["resultados"]) == Q["sha256_resultados"], "Resultados densos alterados.")

RAIZ = next((p for p in (CARPETA, *CARPETA.parents)
             if (p / "06_grafos/catalogo_articulos.csv").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Falta 06_grafos/catalogo_articulos.csv en la raíz del repositorio.")
art = S["artefactos_documentales_existentes"]
r_cat = localizar(["catalogo_articulos.csv"], [RAIZ / "06_grafos"], D["fuentes"]["sha256_catalogo"])
r_npz = localizar([art["npz"]], [RAIZ / "07_RAG"], art["sha256_npz"])
r_idx = localizar([art["indice"]], [RAIZ / "07_RAG"], art["sha256_indice"])
RUTAS = dict(seleccion=r_s, evaluacion=r_e, densa=r_d, anotaciones=r_a,
             catalogo=r_cat, npz=r_npz, faiss=r_idx)
HUELLAS = {k: sha(p) for k, p in RUTAS.items()}
cat = pd.read_csv(r_cat, dtype=str, keep_default_na=False).set_index("article_id", verify_integrity=True)
with np.load(r_npz, allow_pickle=False) as z:
    X = np.array(z["vectores"], dtype=np.float32, order="C", copy=True)
    IDS = z["article_ids"].astype(str)
    META = {k: z[k].item() for k in z.files if z[k].shape == ()}
exigir(X.shape == (519, 2560) and np.isfinite(X).all(), "Matriz documental inesperada.")
exigir(len(set(IDS)) == 519 and set(IDS) == set(cat.index), "Identificadores incompatibles.")
exigir(META["modelo_revision"] == CFG["revision"] and META["prefijo_consulta"] == CFG["prefijo_consulta"],
       "Revisión o prefijo diferente al utilizado para los artículos.")
exigir(np.allclose(np.linalg.norm(X, axis=1), 1, atol=1e-5, rtol=0), "Vectores sin normalización L2.")
ID_NUM = np.array([int(a) for a in IDS])
BASE_Q = {q["query_id"]: q for q in D["consultas"]}
BASE_R = {q["query_id"]: q for q in Q["resultados"]}
exigir(len(BASE_Q) == len(BASE_R) == 24 and set(BASE_Q) == set(BASE_R), "Se esperan las 24 consultas originales.")
QRELS = {n: {} for n in A["base"]["necesidades"]}
for c in A["base"]["casos"]:
    QRELS[c["necesidad_id"]][c["article_id"]] = A["anotaciones"][c["caso_id"]]["asistida"]["grado"]
print("Entradas compatibles | 519 artículos | 24 consultas | anotaciones asistidas, no humanas")

## 2. Casos fijados antes de ejecutar
Las reformulaciones fueron redactadas por el asistente para conservar la intención; su
equivalencia no ha sido validada por humanos. Los temas nuevos **no heredan** etiquetas.
Un error tipográfico consiste en intercambiar dos letras internas de una palabra, de forma
reproducible. No se añaden títulos, autores, identificadores ni respuestas a la entrada.

In [ ]:
PARAFRASIS = {
    "N01": (
        "Busco investigaciones sobre preguntas en lenguaje natural a la documentación técnica de un sistema, cuyas respuestas se apoyen en material recuperado.",
        "I need studies on asking natural-language questions about a system's technical documentation, with answers supported by retrieved material."),
    "N02": (
        "¿Qué recursos o técnicas de procesamiento del lenguaje sirven para organizar o clasificar informes mamográficos escritos en español utilizando BI-RADS?",
        "Which language-processing resources or techniques can organize or classify mammography reports written in Spanish using BI-RADS?"),
    "N03": (
        "Busco análisis de las garantías de privacidad cuando se utiliza descenso de gradiente para entrenar y se aplica recorte de gradientes.",
        "I need analyses of privacy guarantees when training uses gradient descent together with gradient clipping."),
    "N04": (
        "¿Qué herramientas transforman documentos de texto en grafos utilizables para recuperar información o clasificar temas?",
        "Which tools transform text documents into graphs usable for information retrieval or topic classification?"),
    "N05": (
        "Busco estudios sobre la creación automática de programas desde requisitos en lenguaje natural y sobre los problemas que plantea ese proceso.",
        "I need studies on automatically creating programs from natural-language requirements and on the difficulties involved in that process."),
    "N06": (
        "¿De qué manera se evalúa si conviene integrar datos de diferentes fuentes para mejorar la estimación del esfuerzo de desarrollo de software?",
        "How can one assess whether integrating data from different sources is appropriate for improving software development effort estimation?"),
    "N07": (
        "¿Qué plataformas procesan mediciones de sistemas eléctricos para estudiar tanto las oscilaciones como la agrupación de nodos con comportamiento coherente?",
        "Which platforms process power-system measurements to study both oscillations and groups of nodes exhibiting coherent behavior?"),
    "N08": (
        "Busco trabajos que apliquen modelos neuronales de ecuaciones diferenciales a la predicción de la estructura de enanas blancas y examinen los fallos del pronóstico.",
        "I need studies applying neural differential-equation models to predict white-dwarf structure and examining forecasting failures."),
    "N09": (
        "¿Cómo se integra el criterio ambiental, social y de gobernanza en el compromiso entre riesgo y rendimiento al construir un portafolio?",
        "How is the environmental, social, and governance criterion integrated into the risk-return trade-off when constructing a portfolio?"),
    "N10": (
        "Busco métodos que aprovechen la información topológica de estructuras de orden superior para mejorar el aprendizaje contrastivo sobre grafos.",
        "I need methods that use topological information from higher-order structures to improve contrastive learning on graphs."),
    "N11": (
        "¿Qué técnicas de ocultamiento de información en imágenes emplean una clave secreta y procesamiento paralelo para acelerar la codificación?",
        "Which image data-hiding techniques use a secret key and parallel processing to accelerate encoding?"),
    "N12": (
        "Busco estudios del número de policías requerido para capturar ladrones en grafos de fichas construidos a partir de caminos o estrellas.",
        "I need studies of the number of cops required to capture robbers on token graphs constructed from paths or stars.")
}
NUEVAS = {
    "V01": (
        "Busco métodos para extraer o clasificar información de informes clínicos escritos en español; no busco análisis exclusivo de imágenes médicas.",
        "I need methods for extracting or classifying information from clinical reports written in Spanish, not methods exclusively analyzing medical images."),
    "V02": (
        "Busco estudios que representen la conectividad cerebral mediante grafos o hipergrafos construidos con señales EEG o MEG para analizar interacciones de orden superior.",
        "I need studies representing brain connectivity through graphs or hypergraphs built from EEG or MEG signals to analyze higher-order interactions."),
    "V03": (
        "¿Qué estrategias mejoran la clasificación cuando hay pocos datos etiquetados mediante adaptación de dominio, aprendizaje semisupervisado o datos sintéticos?",
        "Which strategies improve classification with limited labeled data through domain adaptation, semi-supervised learning, or synthetic data?"),
    "V04": (
        "Busco investigaciones que utilicen publicaciones y grafos de coautoría para predecir posibles colaboraciones entre investigadores.",
        "I need studies using publications and coauthorship graphs to predict possible collaborations between researchers.")
}
CONTROLES = {
    "C01": ("Busco recetas paso a paso para preparar pan de masa madre en un horno doméstico.",
            "I need step-by-step recipes for baking sourdough bread in a home oven."),
    "C02": ("Busco ediciones críticas de sonetos del Renacimiento italiano con análisis de su métrica.",
            "I need critical editions of Italian Renaissance sonnets with analyses of their meter.")
}

RUBRICAS_NUEVAS = {
    "V01": {
        "2": "Extrae, estructura o clasifica contenido de informes clínicos textuales en español; el idioma está acreditado en título o abstract.",
        "1": "Aporta un procedimiento concreto de PLN sobre informes clínicos en un idioma explícitamente distinto, con utilidad metodológica para la necesidad.",
        "0": "Solo analiza imágenes, señales, datos tabulares o textos no clínicos sin aportar el procedimiento solicitado.",
        "null": "No se puede comprobar el idioma o la naturaleza de los datos; no se infiere español por la institución."},
    "V02": {
        "2": "Construye o estudia conectividad cerebral de orden superior con grafos o hipergrafos derivados de EEG o MEG.",
        "1": "Aporta construcción o análisis concreto de conectividad con EEG/MEG, pero solo acredita interacciones por pares u otro componente del problema.",
        "0": "No aporta conectividad cerebral con las señales solicitadas; compartir grafos, topología o neuronas no basta.",
        "null": "El texto no permite determinar la señal, la representación o el tipo de interacción necesario."},
    "V03": {
        "2": "Estudia clasificación con etiquetas limitadas y utiliza adaptación de dominio, aprendizaje semisupervisado o datos sintéticos.",
        "1": "Aporta un componente concreto transferible a esa clasificación, pero no acredita conjuntamente el escenario de pocas etiquetas y su evaluación clasificatoria.",
        "0": "No aborda una de las estrategias solicitadas ni una contribución concreta a clasificación con etiquetas limitadas.",
        "null": "Los datos disponibles no permiten distinguir el escenario, la tarea o la estrategia."},
    "V04": {
        "2": "Aborda predicción de colaboraciones entre investigadores basada en publicaciones y en un grafo de coautoría; incluye estudios de preparación o evaluación explícita de esa tarea.",
        "1": "Aporta construcción o análisis concreto de coautoría desde publicaciones, pero no acredita una predicción de colaboraciones.",
        "0": "Trata colaboración en otro dominio o predicción de enlaces sin conexión concreta con publicaciones y coautoría académica.",
        "null": "No se puede determinar que el grafo o la predicción correspondan a coautoría."},
    "C01": {
        "2": "Aporta una receta o procedimiento explícito para elaborar pan de masa madre en horno doméstico.",
        "1": "Aporta un paso o técnica concreta de ese proceso de elaboración, aunque no una receta completa.",
        "0": "No aporta el procedimiento culinario solicitado; la coincidencia léxica o temática aislada no basta.",
        "null": "No hay información suficiente para establecer si contiene el procedimiento."},
    "C02": {
        "2": "Presenta o analiza una edición crítica de sonetos del Renacimiento italiano y su métrica.",
        "1": "Aporta análisis métrico concreto de esos sonetos u otro componente explícito de su edición crítica.",
        "0": "No aporta a esa necesidad editorial y literaria; referencias genéricas a lenguaje o patrones no bastan.",
        "null": "No se pueden confirmar el período, el corpus literario o la contribución."}
}


def errata(texto):
    for m in re.finditer(r"[^\W\d_]{7,}", texto, re.UNICODE):
        palabra = m.group()
        if palabra.isupper():
            continue
        i = len(palabra) // 2
        if palabra[i] != palabra[i + 1]:
            nueva = palabra[:i] + palabra[i + 1] + palabra[i] + palabra[i + 2:]
            return texto[:m.start()] + nueva + texto[m.end():], {"original": palabra, "alterada": nueva}
    raise ValueError("No se pudo producir una errata interna no idéntica.")


PRUEBAS = []
for qid, q in BASE_Q.items():
    n, lang = q["necesidad_id"], q["idioma"]
    for bloque, texto, cambio in [
        ("repeticion", q["consulta"], None),
        ("parafrasis", PARAFRASIS[n][0 if lang == "es" else 1], None),
        ("errata", *errata(q["consulta"]))]:
        PRUEBAS.append(dict(prueba_id=f"{qid}-{bloque}", bloque=bloque, necesidad_id=n,
            idioma=lang, division=q["division"], consulta=texto, query_id_base=qid, cambio=cambio))
for bloque, temas in [("necesidad_nueva", NUEVAS), ("control_alcance", CONTROLES)]:
    for n, pareja in temas.items():
        for lang, texto in zip(("es", "en"), pareja):
            PRUEBAS.append(dict(prueba_id=f"{n}-{lang.upper()}", bloque=bloque,
                necesidad_id=n, idioma=lang, division="sondeo_adicional", consulta=texto,
                query_id_base=None, cambio=None))
exigir(len(PRUEBAS) == 84 and len({p["prueba_id"] for p in PRUEBAS}) == 84, "Casos duplicados o faltantes.")
exigir(all(p["consulta"].strip() for p in PRUEBAS), "Consulta vacía.")

def resumen_posiciones(documentos, base_documentos, k):
    nuevos = [d["article_id"] for d in documentos[:k]]
    previos = [d["article_id"] for d in base_documentos[:k]]
    a, b = set(nuevos), set(previos)
    return {"k": k, "retencion_topk": len(a & b) / k,
            "jaccard_topk": len(a & b) / len(a | b),
            "mismo_orden_topk": nuevos == previos}


def diagnostico_etiquetas(necesidad, documentos, k):
    juicios = QRELS.get(necesidad)
    if juicios is None:
        return None
    grados = [juicios.get(d["article_id"]) for d in documentos[:k]]
    positivos = sum(g is not None and g >= 1 for g in grados)
    desconocidos = sum(g is None for g in grados)
    return {"k": k, "con_grado": k - desconocidos, "sin_grado": desconocidos,
            "positivos_identificados": positivos,
            "precision_inf_condicional": positivos / k,
            "precision_sup_condicional": (positivos + desconocidos) / k,
            "nota": "Cotas respecto de etiquetas asistidas y equivalencia asumida; no son IC."}


def guardar(obj):
    temporal = CARPETA / (SALIDA + ".tmp")
    temporal.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporal.replace(CARPETA / SALIDA)


r_libreta = CARPETA / "22_validacion_adicional_qwen3_4b.ipynb"
if not r_libreta.is_file():
    raise FileNotFoundError("Guarda esta libreta como 22_validacion_adicional_qwen3_4b.ipynb antes de ejecutarla.")
nb = json.loads(r_libreta.read_text(encoding="utf-8"))
SHA_CODIGO = firma([c["source"] for c in nb["cells"] if c["cell_type"] == "code"])
CONDICION = dict(version=VERSION, fuentes=HUELLAS, configuracion=CFG, pruebas=PRUEBAS,
    sha256_codigo=SHA_CODIGO, rubricas_nuevas=RUBRICAS_NUEVAS, tolerancia_numerica=TOL_SCORE, batch_size=1,
    equivalencia_variantes="propuesta_por_asistente_no_validada_por_humanos",
    cambio_entorno_permitido=PERMITIR_CAMBIO_ENTORNO)
FIRMA = firma(CONDICION)
r_salida = CARPETA / SALIDA
if r_salida.is_file():
    R = json.loads(r_salida.read_text(encoding="utf-8"))
    exigir(R["firma_condicion"] == FIRMA and R["condicion"] == CONDICION,
           "La salida corresponde a otra condición. Consérvala antes de iniciar una versión distinta.")
    for registro in R["resultados"].values():
        exigir(firma(registro["datos"]) == registro["firma"], "Un resultado guardado fue modificado.")
else:
    R = dict(version=VERSION, firma_condicion=FIRMA, condicion=CONDICION, auditoria={},
             resultados={}, sesiones=[], revision_adicional={}, estado="preparado",
             advertencias=["No es validación humana ni prueba de superioridad entre codificadores.",
                "Las variantes son dependientes de sus necesidades originales.",
                "Nuevas necesidades y controles carecen de juicios de pertinencia.",
                "Un ranking estable no demuestra pertinencia. Coseno no es probabilidad.",
                "No se confirma ausencia de evidencia ni se calibra un umbral de rechazo.",
                "Las latencias adicionales tienen una sola medición, no sustituyen el benchmark previo."])
    guardar(R)
display(pd.DataFrame(PRUEBAS).groupby("bloque").size().rename("consultas").to_frame())
print("Consultas totales: 84 | Originales repetidas una vez: 24 | Adicionales: 60")

## 3. Auditoría del índice y de los resultados existentes (CPU)
Se reconstruyen vectores **leyendo** FAISS; no se calculan embeddings documentales.
Se contrasta FAISS con producto interno directo y con los puntajes guardados. Diferencias
muy pequeñas o empates pueden cambiar un orden; se registran sin confundirlos con calidad.

In [ ]:
import faiss
faiss.omp_set_num_threads(1)
indice = faiss.read_index(str(r_idx))
exigir(type(indice).__name__ == "IndexFlatIP", "El índice no es IndexFlatIP.")
exigir(indice.ntotal == len(IDS) and indice.d == X.shape[1], "Índice y matriz incompatibles.")
exigir(np.array_equal(indice.reconstruct_n(0, len(IDS)), X), "Los vectores del índice no coinciden con el NPZ.")


def buscar(v):
    v = np.asarray(v, dtype=np.float32).reshape(1, -1)
    exigir(v.shape == (1, X.shape[1]) and np.isfinite(v).all(), "Vector de consulta inválido.")
    scores, posiciones = indice.search(np.ascontiguousarray(v), len(IDS))
    scores, posiciones = scores[0], posiciones[0]
    exigir(set(posiciones.tolist()) == set(range(len(IDS))), "FAISS no devolvió cada artículo exactamente una vez.")
    orden = np.lexsort((ID_NUM[posiciones], -scores))
    return [{"article_id": str(IDS[posiciones[j]]), "score_coseno": float(scores[j]), "rank": i + 1}
            for i, j in enumerate(orden)]


auditoria_consultas = []
for qid, previo in BASE_R.items():
    v = np.array(previo["vector_consulta"], dtype=np.float32)
    exigir(np.isclose(np.linalg.norm(v), 1, atol=1e-5, rtol=0), f"Consulta no normalizada: {qid}")
    actual = buscar(v)
    directos = X @ v
    faiss_por_id = {d["article_id"]: d["score_coseno"] for d in actual}
    guardados = {d["article_id"]: d["score_coseno"] for d in previo["documentos"]}
    exigir(set(guardados) == set(IDS), f"Ranking incompleto: {qid}")
    error_matriz = max(abs(faiss_por_id[a] - float(directos[i])) for i, a in enumerate(IDS))
    error_guardado = max(abs(faiss_por_id[a] - guardados[a]) for a in IDS)
    exigir(error_matriz <= TOL_SCORE and error_guardado <= TOL_SCORE,
           f"Puntajes incompatibles para {qid}: matriz={error_matriz}, guardado={error_guardado}")
    auditoria_consultas.append(dict(query_id=qid, max_error_matriz=error_matriz,
        max_error_guardado=error_guardado,
        mismo_top20=[d["article_id"] for d in actual[:20]] == [d["article_id"] for d in previo["documentos"][:20]]))
R["auditoria"] = dict(n_articulos=len(IDS), dimension=X.shape[1], vectores_npz_faiss_identicos=True,
    max_error_norma=float(np.max(np.abs(np.linalg.norm(X, axis=1)-1))), consultas=auditoria_consultas,
    articulos_recalculados=0)
guardar(R)
print("NPZ e índice idénticos | 24 rankings contrastados | 0 embeddings documentales recalculados")
print("Mayor diferencia numérica:", max(a["max_error_matriz"] for a in auditoria_consultas))

## 4. Codificación y búsqueda (GPU)
Carga exclusivamente la revisión y precisión evaluadas, con el mismo `prompt_name="query"`.
Reutiliza resultados completos de esta validación al reanudar. La primera consulta se usa
una vez para calentamiento; después se realiza una medición por caso, no diez.

Si cambió el entorno, se detiene: no actualices paquetes para ejecutar esta prueba.
`PERMITIR_CAMBIO_ENTORNO=True` documentaría una condición distinta, no una réplica estricta.
No se detiene Ollama automáticamente; descarga otros modelos de la GPU antes de ejecutar.

In [ ]:
pendientes = [p for p in PRUEBAS if p["prueba_id"] not in R["resultados"]]
if not pendientes:
    print("Los 84 casos ya están guardados; no se carga el modelo.")
else:
    import torch, sentence_transformers, transformers
    from sentence_transformers import SentenceTransformer
    torch.set_float32_matmul_precision("highest")
    torch.backends.cuda.matmul.allow_tf32 = False
    exigir(torch.cuda.is_available(), "Se necesita CUDA en el kernel donde generaste los embeddings.")
    entorno = dict(python=platform.python_version(), torch=torch.__version__,
        sentence_transformers=sentence_transformers.__version__, transformers=transformers.__version__,
        numpy=np.__version__, faiss=faiss.__version__, gpu=torch.cuda.get_device_name(0), sistema=platform.platform())
    referencia = Q["ejecuciones"][-1]["entorno"]
    cambios = {k: {"anterior": referencia.get(k), "actual": entorno[k]} for k in
               ("torch", "sentence_transformers", "transformers", "numpy", "faiss")
               if referencia.get(k) != entorno[k]}
    exigir(not cambios or PERMITIR_CAMBIO_ENTORNO, f"Entorno diferente al evaluado: {cambios}")
    libre, total = torch.cuda.mem_get_info()
    print(f"GPU libre: {libre / 2**30:.2f} GiB | Casos pendientes: {len(pendientes)}")
    sesion = dict(inicio_utc=datetime.now(timezone.utc).isoformat(), entorno=entorno,
                  cambios_entorno=cambios, gpu_libre_inicial_bytes=libre,
                  gpu_total_bytes=total, mediciones_por_consulta=1, estado="en_ejecucion")
    R["sesiones"].append(sesion)
    R["estado"] = "en_ejecucion"
    guardar(R)
    modelo = None
    try:
        torch.cuda.reset_peak_memory_stats()
        inicio = time.perf_counter()
        modelo = SentenceTransformer(CFG["modelo"], revision=CFG["revision"], device="cuda",
            local_files_only=SOLO_CACHE_LOCAL, model_kwargs={"torch_dtype": torch.bfloat16})
        modelo.eval()
        modelo.max_seq_length = min(int(modelo.max_seq_length), 32768)
        exigir(modelo.prompts.get("query") == CFG["prefijo_consulta"], "Cambió el prefijo de consulta.")
        exigir(modelo.default_prompt_name is None, "Prompt predeterminado inesperado.")
        get_dim = getattr(modelo, "get_embedding_dimension", None)
        if not callable(get_dim):
            get_dim = modelo.get_sentence_embedding_dimension
        exigir(get_dim() == 2560, "Dimensión incorrecta.")
        dtypes = sorted({str(p.dtype) for p in modelo.parameters() if p.is_floating_point()})
        exigir(dtypes == ["torch.bfloat16"], f"Precisión distinta: {dtypes}")
        torch.cuda.synchronize()
        sesion["carga_modelo_ms"] = (time.perf_counter() - inicio) * 1000
        sesion["max_seq_length"] = int(modelo.max_seq_length)
        sesion["prompt_literal"] = modelo.prompts["query"]
        sesion["dtype_modelo"] = dtypes
        sesion["revision_solicitada"] = CFG["revision"]
        sesion["revision_reportada"] = getattr(modelo[0].auto_model.config, "_commit_hash", None)

        def codificar(texto):
            with torch.inference_mode():
                salida = modelo.encode([texto], prompt_name="query", batch_size=1,
                    normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
            v = np.array(salida, dtype=np.float32, order="C", copy=True)
            exigir(v.shape == (1, 2560) and np.isfinite(v).all(), "Embedding inválido.")
            exigir(np.linalg.norm(v) > 0, "Embedding de norma cero.")
            faiss.normalize_L2(v)
            return v

        codificar(pendientes[0]["consulta"])
        torch.cuda.synchronize()
        for i, p in enumerate(pendientes, 1):
            completo = CFG["prefijo_consulta"] + p["consulta"]
            tokens = len(modelo.tokenizer(completo, truncation=False, add_special_tokens=True)["input_ids"])
            exigir(tokens <= modelo.max_seq_length, f"Consulta excede el límite de contexto: {p['prueba_id']}")
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            v = codificar(p["consulta"])
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            documentos = buscar(v)
            t2 = time.perf_counter()
            datos = dict(**p, tokens_con_prefijo=tokens, vector_consulta=v[0].tolist(), documentos=documentos,
                codificacion_ms=(t1-t0)*1000, busqueda_ms=(t2-t1)*1000, total_ms=(t2-t0)*1000)
            if p["bloque"] == "repeticion":
                anterior = BASE_R[p["query_id_base"]]
                vg = np.array(anterior["vector_consulta"], dtype=np.float32)
                sg = {d["article_id"]: d["score_coseno"] for d in anterior["documentos"]}
                datos["replica"] = dict(max_error_vector=float(np.max(np.abs(v[0]-vg))),
                    max_error_score=max(abs(d["score_coseno"]-sg[d["article_id"]]) for d in documentos),
                    mismo_top20=[d["article_id"] for d in documentos[:20]] == [d["article_id"] for d in anterior["documentos"][:20]])
            R["resultados"][p["prueba_id"]] = {"datos": datos, "firma": firma(datos)}
            guardar(R)
            if i % 12 == 0 or i == len(pendientes):
                print(f"Completados: {len(R['resultados'])}/84")
        sesion["estado"] = "completada"
        R["estado"] = "inferencias_completas"
    except BaseException as exc:
        sesion["estado"] = "interrumpida"
        sesion["error"] = f"{type(exc).__name__}: {exc}"
        R["estado"] = "interrumpido_reanudable"
        raise
    finally:
        sesion["fin_utc"] = datetime.now(timezone.utc).isoformat()
        sesion["pico_pytorch_asignado_bytes"] = torch.cuda.max_memory_allocated()
        sesion["pico_pytorch_reservado_bytes"] = torch.cuda.max_memory_reserved()
        sesion["alcance_memoria"] = "Proceso PyTorch: carga y consultas; no VRAM total ni coexistencia con Gemma."
        if modelo is not None:
            del modelo
        gc.collect()
        torch.cuda.empty_cache()
        guardar(R)

## 5. Diagnóstico y casos que necesitan juicio
Se informa retención y orden de resultados, no una nueva puntuación global de calidad.
Las cotas de Precision dependen de las etiquetas asistidas existentes y de la equivalencia
asumida de cada variante. Un documento fuera del conjunto revisado se mantiene desconocido.
No se calcula nDCG/Recall para los temas nuevos sin sus juicios.

In [ ]:
exigir(len(R["resultados"]) == len(PRUEBAS), "La ejecución está incompleta; reanuda la celda anterior.")
DIAGNOSTICOS, pool = [], {}
for p in PRUEBAS:
    datos = R["resultados"][p["prueba_id"]]["datos"]
    docs = datos["documentos"]
    base_docs = BASE_R[p["query_id_base"]]["documentos"] if p["query_id_base"] else None
    for k in (5, 10, 20):
        fila = dict(prueba_id=p["prueba_id"], bloque=p["bloque"], necesidad_id=p["necesidad_id"],
                    idioma=p["idioma"], division=p["division"], k=k)
        if base_docs is not None:
            fila.update(resumen_posiciones(docs, base_docs, k))
            fila.update(diagnostico_etiquetas(p["necesidad_id"], docs, k))
        textos = [cat.loc[d["article_id"], "abstract"] for d in docs[:k]]
        fila["resumenes_distintos"] = len(set(textos))
        DIAGNOSTICOS.append(fila)
    for d in docs[:20]:
        n, a = p["necesidad_id"], d["article_id"]
        if n in QRELS and a in QRELS[n] and QRELS[n][a] is not None:
            continue
        cid = f"{n}-A{int(a):04d}"
        if cid not in pool:
            consultas = {x["idioma"]: x["consulta"] for x in PRUEBAS
                          if x["necesidad_id"] == n and x["bloque"] == ("repeticion" if n in QRELS else p["bloque"])}
            pool[cid] = dict(caso_id=cid, necesidad_id=n, article_id=a, consultas=consultas,
                titulo=cat.loc[a, "titulo"], abstract=cat.loc[a, "abstract"],
                criterio_original=A["base"]["necesidades"].get(n, {}).get("criterios", RUBRICAS_NUEVAS.get(n)),
                motivo="informacion_insuficiente_previa" if n in QRELS and a in QRELS[n] else "no_juzgado",
                anotacion_asistida={"estado": "pendiente", "grado": None},
                anotacion_humana={"estado": "pendiente", "grado": None})
# Preservar anotaciones adicionales introducidas después de esta ejecución.
for cid, caso in pool.items():
    if cid in R["revision_adicional"]:
        previo = R["revision_adicional"][cid]
        exigir(all(previo[k] == caso[k] for k in ("article_id", "necesidad_id", "titulo", "abstract")),
               f"Cambió la base de revisión adicional: {cid}")
        caso["anotacion_asistida"] = previo.get("anotacion_asistida", caso["anotacion_asistida"])
        caso["anotacion_humana"] = previo.get("anotacion_humana", caso["anotacion_humana"])
R["revision_adicional"].update(pool)
R["diagnosticos"] = DIAGNOSTICOS
R["estado"] = "ejecutado_pendiente_revision_semantica"
R["metricas_calidad_nuevas"] = None
R["seleccion_definitiva_realizada"] = False
exigir(all(sha(p) == HUELLAS[k] for k, p in RUTAS.items()), "Alguna entrada cambió durante la prueba.")
guardar(R)

tabla = pd.DataFrame(DIAGNOSTICOS)
v = tabla.query("k == 10 and bloque in ['parafrasis', 'errata']")
display(v.groupby(["division", "bloque", "idioma"])[
    ["retencion_topk", "jaccard_topk", "con_grado", "sin_grado", "precision_inf_condicional", "precision_sup_condicional"]].mean().round(4))
replicas = [r["datos"]["replica"] for r in R["resultados"].values() if r["datos"]["bloque"] == "repeticion"]
print("Réplicas con top20 idéntico:", sum(r["mismo_top20"] for r in replicas), "/", len(replicas))
print("Mayor variación de score al recodificar:", max(r["max_error_score"] for r in replicas))
print("Pares adicionales que requieren juicio:", len(R["revision_adicional"]))
print("Archivo:", r_salida.resolve())
print("Comparte este JSON. No se ha aprobado un ganador ni asignado irrelevancia a los controles.")

## Interpretación y criterio de cierre

**Integridad:** los archivos y vectores deben corresponder a la condición evaluada. Una
incompatibilidad detiene el proceso; diferencias de réplica o de orden requieren examen.
**Robustez:** revisar pérdidas de evidencia conocida y casos con cambios de orden; alta
retención no basta para afirmar pertinencia. No hay un umbral de aprobación inventado.
**Sondeos:** revisar los resultados de V01–V04 y C01–C02 sin asumir que sus temas tienen o
carecen de respuesta en el catálogo. Las rúbricas nuevas quedan fijadas en la celda 2 antes de la búsqueda; no se ajustan para
favorecer los documentos recuperados. La lista de revisión no muestra posiciones ni puntajes.

Las nuevas anotaciones se conservan aparte. Una revisión asistida seguirá siendo asistida.
La comparación anterior permanece intacta y la elección de Qwen3-4B sigue siendo provisional.
Probar solo Qwen3-4B no demuestra que supere a Nemotron, Arctic o PPLX en estos nuevos casos.
No se validan todavía evidencia de pares de autores ni factores del predictor de coautoría.

**Fuentes técnicas:** configuración oficial de Qwen3-Embedding-4B, revisión
`5cf2132abc99cad020ac570b19d031efec650f2b`; documentación de FAISS `IndexFlat` y
`MetricType and distances`; Manning, Raghavan y Schütze, *Introduction to Information
Retrieval*, secciones 8.1 y 8.5.1. La composición del ensayo es una propuesta para este proyecto,
no un benchmark externo estandarizado.